# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/james-kalaw/FlyRank-ML-Intern-Kalaw/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule idea, in plain words:**
> Review **young pages** whose keyword has **low search volume** first. Among those, put the pages
> with the most February impressions at the top, because they have the most visibility to lose.

- **Reason code:** `young_low_volume`
- **Action label:** `review_for_decline_risk`

**Why these two signals:** in Week 2 (starter CSV), younger pages declined more (gap −16.4 points)
and pages with lower search volume declined more (gap −12.2). **Search volume** is also the signal
behind FlyRank's **quick-win** flag. I do not use staleness (days since the last update): in
Week 3, 76.5% of pages were updated after 1 March, so that date is not known on the decision day.

**Data:** the same Feb → Mar 2026 warehouse frame as Week 3 (features from 1–28 Feb, outcome from
March, same filters). The model in Week 5 will use the same pages and labels, so the comparison is fair.

**Before I build the rule, I check both signals.** Each gets a table: pages split into buckets,
with the number of pages (n) and the % declining in each bucket.

**How I give the verdict (decided before looking):** I only count buckets with at least 500 pages.
- **CONFIRMED:** decline goes the way my rule expects (younger / lower volume → more decline) and
  the two end buckets differ by 5+ points.
- **OPPOSITE:** the two end buckets differ by 5+ points, but the other way.
- **MIXED:** the pattern goes up and down, or the difference is only 2–5 points.
- **FALSE:** every bucket is within 2 points of the base rate.

In [ ]:
# Setup: connect DuckDB to the warehouse (token comes from the Colab Secrets panel, never typed here)
%pip -q install duckdb
import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
# Only the two months I need (a list of paths; hf:// doesn't support {a,b} globs)
FACT_FEB_MAR = (f"read_parquet(['{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
                f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'])")

BEFORE_START, DECISION, AFTER_END = "2026-02-01", "2026-03-01", "2026-03-31"
print('Connected to the warehouse.')

In [ ]:
FEATURES = ["content_age_days", "word_count", "search_volume", "feb_avg_position", "feb_impressions"]

# Build one row per page: Feb features + March outcome
frame = con.sql(f"""
    WITH ok_clients AS (
        SELECT client_hash_id FROM {DIM_CLIENTS}
        WHERE gsc_data_start <= DATE '{BEFORE_START}'
    ),
    per_page AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions) FILTER (WHERE f.report_date <  DATE '{DECISION}') AS feb_impressions,
               SUM(f.gsc_clicks)      FILTER (WHERE f.report_date <  DATE '{DECISION}') AS feb_clicks,
               SUM(f.gsc_avg_position * f.gsc_impressions) FILTER (WHERE f.report_date < DATE '{DECISION}')
                 / NULLIF(SUM(f.gsc_impressions) FILTER (WHERE f.report_date < DATE '{DECISION}'), 0)
                                                                                         AS feb_avg_position,
               COUNT(*) FILTER (WHERE f.report_date >= DATE '{DECISION}')                AS mar_days,
               SUM(f.gsc_impressions) FILTER (WHERE f.report_date >= DATE '{DECISION}')  AS mar_impressions
        FROM {FACT_FEB_MAR} f
        JOIN ok_clients USING (client_hash_id)
        WHERE f.gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT p.client_hash_id, p.content_hash_id,
           date_diff('day', d.content_created_date, DATE '{DECISION}') AS content_age_days,
           d.word_count, d.search_volume, d.content_type,
           p.feb_avg_position, p.feb_impressions, p.feb_clicks, p.mar_days, p.mar_impressions
    FROM per_page p
    JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE d.content_created_date < DATE '{BEFORE_START}'
      AND p.feb_impressions >= 100
""").df()

pages_before = len(frame)
frame = frame[frame["mar_days"] > 0].copy()          # pages with no usable March rows: label unknown
dropped_no_march = pages_before - len(frame)

# Label: March impressions per day more than 20% below February's
feb_per_day = frame["feb_impressions"] / 28
mar_per_day = frame["mar_impressions"] / 31
frame["is_declining"] = (mar_per_day < 0.8 * feb_per_day).astype(int)

print(f"Pages in the frame: {len(frame):,} from {frame['client_hash_id'].nunique()} clients")
print(f"Dropped (no usable March data): {dropped_no_march:,}")
print(f"Base rate: {frame['is_declining'].mean() * 100:.1f}% declining\n")
print("Feature summary (no rows shown):")
print(frame[FEATURES].astype(float).describe().T[["count", "mean", "50%", "min", "max"]].round(1).to_string())
print("\nMissing per feature (%):")
print((frame[FEATURES].isna().mean() * 100).round(1).to_string())

### Signal 1: page age (my rule expects: younger → more decline)

**My verdict (write after Run all):** _CONFIRMED / OPPOSITE / MIXED / FALSE, and why in one line._

In [ ]:
# Bucket table helper: n and % declining per bucket, plus missing values
def bucket_table(col, bins, labels):
    buckets = pd.cut(frame[col], bins=bins, labels=labels, right=False)
    t = frame.groupby(buckets, observed=False)["is_declining"].agg(pages="size", decline_rate="mean")
    missing = frame[col].isna()
    if missing.any():
        t.loc["missing"] = [missing.sum(), frame.loc[missing, "is_declining"].mean()]
    t["pages"] = t["pages"].astype(int)
    t["decline_%"] = (t.pop("decline_rate") * 100).round(1)
    t["vs_base_pts"] = (t["decline_%"] - BASE).round(1)
    return t

def read_table(t):
    big = t.drop(index="missing", errors="ignore")
    big = big[big["pages"] >= 500]
    if len(big) < 2:
        print("\nFewer than 2 buckets have 500+ pages: too small to judge.")
        return
    first, last = big["decline_%"].iloc[0], big["decline_%"].iloc[-1]
    steps = big["decline_%"].diff().dropna()
    one_way = (steps <= 0).all() or (steps >= 0).all()
    print(f"\nBuckets with 500+ pages: {len(big)}")
    print(f"First bucket {first}% -> last bucket {last}%: difference {last - first:+.1f} points")
    print("Goes one way (no ups and downs):", one_way)
    print("Biggest distance from base rate:", f"{big['vs_base_pts'].abs().max():.1f} points")

BASE = frame["is_declining"].mean() * 100
print(f"Base rate: {BASE:.1f}% declining ({len(frame):,} pages)\n")

age = bucket_table("content_age_days",
                   bins=[0, 90, 180, 270, 365, float("inf")],
                   labels=["<90 days", "90-179", "180-269", "270-364", "365+"])
print("Decline by page age (days on 1 March):")
print(age.to_string())
read_table(age)

### Signal 2: search volume (flag-linked: quick-win). My rule expects: lower volume → more decline

**My verdict (write after Run all):** _CONFIRMED / OPPOSITE / MIXED / FALSE, and why in one line._

In [ ]:
volume = bucket_table("search_volume",
                      bins=[0, 1, 11, 101, 1001, float("inf")],
                      labels=["0", "1-10", "11-100", "101-1000", "1001+"])
print("Decline by keyword search volume:")
print(volume.to_string())
read_table(volume)

### What the two checks told me, and what I do next

Both signals the rule idea leaned on came back **FALSE** (every bucket within 2 points of the base rate).
A rule built on them would do no better than picking pages at random. So **I change the rule before
building it.** The card calls this a win: the check saved the rule.

The only bucket far from the base rate was **missing search volume** (1,346 pages, 42.9% declining).
The data dictionary says missing keyword data follows `content_type`, so I check that next.

**Four new clues (with charts), all known on 1 March:**
1. **Content type:** what kind of page it is (explains the "missing" bucket?).
2. **February position:** where the page ranked in February (hill-shaped in Week 2).
3. **CTR vs position (flag-linked: CTR-fix):** does the page get fewer clicks than other pages at the
   same rank? I compare each page's February CTR with the median CTR of pages in its position group.
4. **February impressions:** how visible the page was before 1 March.

**How I pick the clue for my rule (decided before looking):** the clue with a bucket of 500+ pages that is
**farthest from the base rate**, in either direction. If every clue is weak, I still use the strongest one
and say clearly that the baseline is weak. The skill says a baseline should be "honestly beatable."

**Careful:** I tested 2 clues before and 4 now. When you test many clues, one can look strong by luck.
Week 6 checks the chosen clue on other clients and months.

In [ ]:
# Chart helper: % declining per bucket, base rate as a dashed line, n under each bar
import matplotlib.pyplot as plt

def plot_table(t, title):
    t = t.dropna(subset=["decline_%"])
    colors = ["#c0392b" if (n >= 500 and abs(v) >= 5) else "#7f8c8d"
              for n, v in zip(t["pages"], t["vs_base_pts"])]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.bar([str(i) for i in t.index], t["decline_%"], color=colors)
    ax.axhline(BASE, linestyle="--", color="black", linewidth=1)
    ax.text(len(t) - 0.5, BASE + 0.8, f"base rate {BASE:.1f}%", ha="right", fontsize=9)
    for i, (v, n) in enumerate(zip(t["decline_%"], t["pages"])):
        ax.text(i, v + 0.8, f"{v:.1f}%", ha="center", fontsize=9)
        ax.text(i, 1.0, f"n={n:,}", ha="center", fontsize=8, color="white")
    ax.set_ylabel("% declining (Feb -> Mar)")
    ax.set_title(title + "\n(red = 500+ pages and 5+ points from base rate)")
    ax.set_ylim(0, max(55, t["decline_%"].max() + 8))
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

def category_table(col):
    t = frame.groupby(frame[col].fillna("missing"))["is_declining"].agg(pages="size", decline_rate="mean")
    t["decline_%"] = (t.pop("decline_rate") * 100).round(1)
    t["vs_base_pts"] = (t["decline_%"] - BASE).round(1)
    return t.sort_values("pages", ascending=False)

# Charts of the two FALSE checks above, so they can be seen, not just read
plot_table(age, "Signal 1: page age (FALSE)")
plot_table(volume, "Signal 2: search volume (FALSE)")

### New clue 1: content type

**What I see (write after Run all):** _Which content type declines most? Does it explain the missing-volume bucket?_

In [ ]:
ctype = category_table("content_type")
print("Decline by content type:")
print(ctype.to_string())
read_table(ctype.sort_values("decline_%"))   # sorted low -> high, so first/last = the extremes
plot_table(ctype, "New clue 1: content type")

# Where do the missing-search-volume pages come from?
miss = frame["search_volume"].isna()
print("\nPages with missing search volume, by content type:")
print(frame.loc[miss, "content_type"].fillna("missing").value_counts().to_string())

### New clue 2: February position

`0` means "no position data" (no real page ranks 0). I keep it as its own bucket so it can't hide in "top 3".

**What I see (write after Run all):** _Is it hill-shaped like Week 2, a straight line, or flat?_

In [ ]:
position = bucket_table("feb_avg_position",
                        bins=[0, 1, 3.5, 10.5, 20.5, 50.5, float("inf")],
                        labels=["0 (no data)", "1-3", "4-10", "11-20", "21-50", "51+"])
print("Decline by February average position:")
print(position.to_string())
read_table(position)
plot_table(position, "New clue 2: February position")

### New clue 3: CTR vs position (flag-linked: CTR-fix)

**In plain words:** pages at rank 3 usually get more clicks than pages at rank 30, so I can't compare raw
CTR. Instead, I compare each page with **pages at the same rank**:

> `ctr_vs_peers` = the page's February CTR ÷ the CTR of all pages in its position group together
> (all their clicks ÷ all their impressions)

- **Below 1** = fewer clicks than its peers (what FlyRank's CTR-fix logic looks for)
- **Above 1** = more clicks than its peers

**What I see (write after Run all):** _Do pages with fewer clicks than their peers decline more?_

In [ ]:
frame["feb_ctr"] = frame["feb_clicks"] / frame["feb_impressions"] * 100
pos_group = pd.cut(frame["feb_avg_position"], bins=[0, 1, 3.5, 10.5, 20.5, 50.5, float("inf")], right=False)
# Peer CTR = all clicks / all impressions of pages in the same position group
grp = frame.groupby(pos_group, observed=False)
peer_ctr = grp["feb_clicks"].transform("sum") / grp["feb_impressions"].transform("sum") * 100
frame["ctr_vs_peers"] = (frame["feb_ctr"] / peer_ctr).where(peer_ctr > 0)
print("Peer CTR (%) per position group:")
print(frame.groupby(pos_group, observed=False).apply(
    lambda g: round(g["feb_clicks"].sum() / g["feb_impressions"].sum() * 100, 2)
    if g["feb_impressions"].sum() > 0 else None).to_string())
print()

ctr_tab = bucket_table("ctr_vs_peers",
                       bins=[0, 0.5, 0.9, 1.1, 2, float("inf")],
                       labels=["<0.5 (far below peers)", "0.5-0.9", "0.9-1.1 (like peers)", "1.1-2", "2+ (far above)"])
print("Decline by CTR compared with pages at the same position:")
print(ctr_tab.to_string())
read_table(ctr_tab)
plot_table(ctr_tab, "New clue 3: CTR vs position peers")

### New clue 4: February impressions

**What I see (write after Run all):** _Do more visible pages decline more, less, or the same?_

In [ ]:
impressions = bucket_table("feb_impressions",
                           bins=[100, 250, 500, 1000, 2500, 10000, float("inf")],
                           labels=["100-249", "250-499", "500-999", "1k-2.5k", "2.5k-10k", "10k+"])
print("Decline by February impressions:")
print(impressions.to_string())
read_table(impressions)
plot_table(impressions, "New clue 4: February impressions")

### Which clue is strongest? (the rule I set before looking)

**My pick (write after Run all):** _Which clue and which bucket will my rule use, and how strong is it?_

In [ ]:
# For each clue: the bucket (500+ pages) farthest from the base rate
def strongest(name, t):
    big = t[(t["pages"] >= 500)].dropna(subset=["decline_%"])
    if big.empty:
        return {"clue": name, "bucket": "(no bucket with 500+ pages)", "pages": 0,
                "decline_%": float("nan"), "vs_base_pts": 0.0}
    row = big.loc[big["vs_base_pts"].abs().idxmax()]
    return {"clue": name, "bucket": row.name, "pages": int(row["pages"]),
            "decline_%": row["decline_%"], "vs_base_pts": row["vs_base_pts"]}

summary = pd.DataFrame([
    strongest("page age", age),
    strongest("search volume", volume),
    strongest("content type", ctype),
    strongest("February position", position),
    strongest("CTR vs position", ctr_tab),
    strongest("February impressions", impressions),
])
summary["distance"] = summary["vs_base_pts"].abs()
summary = summary.sort_values("distance", ascending=False).drop(columns="distance")
print(f"Base rate: {BASE:.1f}%  |  bucket farthest from it, per clue (500+ pages only):\n")
print(summary.to_string(index=False))
print(f"\nStrongest clue: {summary.iloc[0]['clue']}  (bucket '{summary.iloc[0]['bucket']}', "
      f"{summary.iloc[0]['vs_base_pts']:+.1f} points)")

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**My final rule, in plain words:**
> Review **feedly articles** first, starting with the ones that had the **most February impressions**
> (they have the most visibility to lose). Every other page gets score 0.

- **Score:** `is_feedly × feb_impressions` (no fitted weights, readable on purpose)
- **Reason code:** `feedly_article`
- **Action label:** `review_for_decline_risk`

**Why this rule:** both signals from my first idea (page age, search volume) were FALSE. Using the selection
rule I wrote before looking, the strongest clue was content type: feedly articles declined at about double
the base rate. February impressions only decide the order *inside* the feedly group.

**How I score the rule:** precision@K, meaning: of the top K pages the rule picks, how many really declined?
I print it next to the **base rate** and next to **random picking** (shuffle the pages, take the top K,
repeat 20 times). The rule is only useful if it beats both.

**A limit I know in advance:** there are only a few hundred feedly articles. After them, every page has
score 0, so the rule can't rank the rest. It is only useful for the top of the list.

**My result (write after Run all):** _precision@10/50/100/500 vs the base rate, and how many clients the feedly pages come from._

In [ ]:
# The rule: feedly articles first, ordered by February impressions
import json, os
import numpy as np

FEEDLY = "feedly article"
RULE_COLUMNS = ["content_type", "feb_impressions"]       # everything the rule reads

q = frame.copy()
q["is_feedly"] = (q["content_type"] == FEEDLY).astype(int)
q["score"] = q["is_feedly"] * q["feb_impressions"]
q["reason_code"] = np.where(q["is_feedly"] == 1, "feedly_article", "none")
q["action"] = np.where(q["is_feedly"] == 1, "review_for_decline_risk", "no_action")
q = q.sort_values(["score", "feb_impressions"], ascending=False).reset_index(drop=True)
q["rank"] = q.index + 1

# Pseudonymous client labels for display: "client 1" = the client with the most pages
size_order = frame["client_hash_id"].value_counts().index
client_label = {c: f"client {i + 1}" for i, c in enumerate(size_order)}
q["client"] = q["client_hash_id"].map(client_label)

# Write the ranked queue (work/**/*.csv is git-ignored by design)
os.makedirs("work/outputs", exist_ok=True)
q[["rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False)
n_flagged = int(q["is_feedly"].sum())
print(f"Wrote work/outputs/baseline_action_score.csv: {len(q):,} pages ranked, {n_flagged:,} flagged by the rule")

# precision@K for the rule vs. random picking
def precision_at_k(labels_in_rank_order, k):
    return float(np.asarray(labels_in_rank_order)[:k].mean())

rng = np.random.default_rng(42)
labels = q["is_declining"].to_numpy()
rows = []
for k in (10, 50, 100, 500):
    rand = np.mean([precision_at_k(rng.permutation(labels), k) for _ in range(20)])
    rows.append({"K": k, "rule_precision_%": round(precision_at_k(labels, k) * 100, 1),
                 "random_picking_%": round(rand * 100, 1), "base_rate_%": round(BASE, 1),
                 "all_K_flagged_by_rule": k <= n_flagged})
prec = pd.DataFrame(rows)
print("\nPrecision@K (of the top K, how many really declined):")
print(prec.to_string(index=False))

# Where do the flagged (feedly) pages come from?
flagged = q[q["is_feedly"] == 1]
per_client = flagged["client"].value_counts()
print(f"\nFlagged pages come from {per_client.size} client(s).")
print(f"Biggest client share of flagged pages: {per_client.iloc[0] / len(flagged):.1%} ({per_client.index[0]})")
print("Flagged pages per client (top 5):")
print(per_client.head().to_string())

# Metrics receipt (small JSON, safe to commit)
metrics = {"rule": "feedly_article x feb_impressions", "pages": int(len(q)), "flagged": n_flagged,
           "base_rate_pct": round(BASE, 1),
           "precision_pct": {f"@{r['K']}": r["rule_precision_%"] for r in rows},
           "random_pct": {f"@{r['K']}": r["random_picking_%"] for r in rows},
           "flagged_clients": int(per_client.size),
           "biggest_client_share_pct": round(per_client.iloc[0] / len(flagged) * 100, 1)}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nwork/outputs/baseline_metrics.json:")
print(json.dumps(metrics, indent=2))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The card asks for the **top 10** (the skeleton says 20). For each page: the action, why it is there, and what
would make it wrong. Pages are shown by rank and a client number, never by their IDs.

**What the columns mean:**
- `declined_in_march`: did the page really decline? (1 = yes). This is the answer, shown **only to check the
  rule**; the rule never saw it.
- `ctr_vs_peers`: clicks compared with pages at the same rank (below 1 = fewer clicks than peers)
- `what_could_make_it_wrong`: reasons this pick might be a mistake, built from the clues I already checked

**My result (write after Run all):** _How many of the top 10 declined? What do the wrong picks have in common?_

In [ ]:
# Top-10 review: one line per page
top = q.head(10).copy()

def why(r):
    return f"feedly article with {int(r['feb_impressions']):,} Feb impressions (rank {r['rank']} by visibility)"

def could_be_wrong(r):
    notes = []
    if per_client.iloc[0] / len(flagged) >= 0.5 and r["client"] == per_client.index[0]:
        notes.append(f"{r['client']} holds most feedly pages: maybe a client habit, not a content-type effect")
    if r["feb_avg_position"] == 0:
        notes.append("position 0 = no position data (data quirk)")
    elif r["feb_avg_position"] > 20.5:
        notes.append(f"ranks deep (avg {r['feb_avg_position']:.0f}): deep pages declined less ({position.loc['21-50', 'decline_%']}% for 21-50)")
    if pd.notna(r.get("ctr_vs_peers")) and r["ctr_vs_peers"] >= 1.1:
        notes.append(f"gets more clicks than its peers ({r['ctr_vs_peers']:.1f}x): those pages declined less")
    notes.append("a February spike that is just going back to normal (noise/seasonality) looks the same")
    notes.append("traffic may have moved to a sibling page (consolidation), not checked")
    return "; ".join(notes)

top["action"] = "review_for_decline_risk"
top["why_it_is_here"] = top.apply(why, axis=1)
top["what_could_make_it_wrong"] = top.apply(could_be_wrong, axis=1)
top["declined_in_march"] = top["is_declining"]

view = top[["rank", "client", "feb_impressions", "feb_avg_position", "ctr_vs_peers", "declined_in_march"]].copy()
view["feb_avg_position"] = view["feb_avg_position"].round(1)
view["ctr_vs_peers"] = view["ctr_vs_peers"].round(2)
print(view.to_string(index=False))
print(f"\nTop 10: {int(top['declined_in_march'].sum())} of 10 really declined (base rate {BASE:.1f}%)\n")

for _, r in top.iterrows():
    print(f"#{r['rank']:>2} | action: {r['action']} | why: {r['why_it_is_here']}")
    print(f"    | what could make it wrong: {r['what_could_make_it_wrong']}")

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** pages the rule put near the top that did **not** decline. I compare them with the picks that did,
to see what the wrong ones have in common.

**Leakage check:** the code checks that the rule only reads `content_type` and `feb_impressions`, never March
data, the label, or the excluded columns from my Week 3 contract (`content_updated_date`,
`last_optimized_date`, `is_deleted`, `is_published`, product metadata). FlyRank's product flags
(`health_score`, `priority_score`, `action_type`) are not in this data at all.

One honest caveat: `content_type` comes from `dim_content`, a present-day snapshot. A page's type is unlikely to
change after it is published, so I treat it as known on 1 March, but I can't prove it from this data.

**My result (write after Run all):** _What do the weak picks have in common? Did the leakage check pass?_

In [ ]:
# Weak picks: among the flagged (feedly) pages, what separates the misses from the hits?
hits = flagged[flagged["is_declining"] == 1]
misses = flagged[flagged["is_declining"] == 0]
compare = pd.DataFrame({
    "declined (hits)": [len(hits), hits["feb_impressions"].median(), hits["feb_avg_position"].median(),
                        hits["ctr_vs_peers"].median(), hits["content_age_days"].median()],
    "did not decline (misses)": [len(misses), misses["feb_impressions"].median(), misses["feb_avg_position"].median(),
                                 misses["ctr_vs_peers"].median(), misses["content_age_days"].median()],
}, index=["pages", "median Feb impressions", "median Feb position", "median CTR vs peers", "median age (days)"]).round(2)
print(f"Flagged pages: {len(flagged):,} | hits {len(hits):,} | misses {len(misses):,} "
      f"({len(hits) / len(flagged):.1%} precision over all flagged pages)\n")
print(compare.to_string())

# Leakage check: the rule may only read allowed February-or-earlier fields
FORBIDDEN = {"mar_impressions", "mar_days", "is_declining", "content_updated_date", "last_optimized_date",
             "is_deleted", "is_published", "provider_used", "model_used",
             "health_score", "priority_score", "action_type"}
assert set(RULE_COLUMNS) <= set(FEATURES + ["content_type"]), "rule reads a column outside the contract"
assert not set(RULE_COLUMNS) & FORBIDDEN, "rule reads a label, future, or excluded column"
assert not {"health_score", "priority_score", "action_type"} & set(frame.columns), "product flag found in data"
print("\nLeakage check passed: the rule reads only", RULE_COLUMNS,
      "| no March data, label, excluded column, or product flag.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.